# Finding the Missing Profiling Step X

The objective of this notebook is to reverse-engineer the unknown filtering step that reduced
the daily profile population from approximately 24k profiles to approximately 9–10k profiles.

We compare the Day-15 profiles generated by our reproduction of the team lead's profiling function
against the corresponding Day-15 dataset produced by the team lead, using population, feature,
distribution, threshold, and longitudinal diagnostics.

---

### Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sweetviz as sv

from pathlib import Path

### Paths

In [4]:
PROJECT_ROOT = Path.cwd().parent.parent

DATA_PATH = PROJECT_ROOT / "notebooks" / "troubleshooting"

MY_DAY15_PATH = DATA_PATH / "df_11_15_CM_fraud_af.csv"
LEAD_DAY15_PATH = DATA_PATH / "df_11_15_CM_fraud.csv"

ALL_PROFILES_PATH = PROJECT_ROOT / "data" / "profiles" / "all_profiles.parquet"

### Load both datasets

In [5]:
my_day15 = pd.read_csv(MY_DAY15_PATH)
lead_day15 = pd.read_csv(LEAD_DAY15_PATH)

print("My Day 15:", my_day15.shape)
print("Lead Day 15:", lead_day15.shape)

My Day 15: (23541, 24)
Lead Day 15: (9581, 25)


### Inspect columns

In [6]:
print("My columns:")
print(my_day15.columns.tolist())

print("\nLead columns:")
print(lead_day15.columns.tolist())

My columns:
['nb_out_calls', 'nb_in_calls', 'nb_called_numbers', 'avg_out_duration', 'avg_in_duration', 'ratio_in_vs_out', 'ratio_distinct_vs_out', 'ratio_in_duration_vs_out_duration', 'dawn_count', 'day_count', 'afternoon_count', 'night_count', 'ratio_dawn_vs_out', 'ratio_day_vs_out', 'ratio_afternoon_vs_out', 'ratio_night_vs_out', 'nb_flagged_calls', 'ratio_flagged_vs_out', 'nb_distinct_lacs', 'ratio_distinct_lacs_vs_out', 'nb_distinct_cell_ids', 'ratio_distinct_cell_ids_vs_out', 'avg_delay_per_day', 'fraud']

Lead columns:
['calling_number', 'nb_out_calls', 'nb_in_calls', 'nb_called_numbers', 'avg_out_duration', 'avg_in_duration', 'ratio_in_vs_out', 'ratio_distinct_vs_out', 'ratio_in_duration_vs_out_duration', 'dawn_count', 'day_count', 'afternoon_count', 'night_count', 'ratio_dawn_vs_out', 'ratio_day_vs_out', 'ratio_afternoon_vs_out', 'ratio_night_vs_out', 'nb_flagged_calls', 'ratio_flagged_vs_out', 'nb_distinct_lacs', 'ratio_distinct_lacs_vs_out', 'nb_distinct_cell_ids', 'ratio_di

### Normalize the profile identifier

In [7]:
def normalize_number(series):
    return (
        pd.to_numeric(series, errors="coerce")
        .dropna()
        .astype("int64")
        .astype(str)
    )


my_day15["calling_number_normalized"] = normalize_number(
    my_day15["calling_number"]
)

lead_day15["calling_number_normalized"] = normalize_number(
    lead_day15["calling_number"]
)

KeyError: 'calling_number'

### Basic population comparison

In [ ]:
my_ids = set(my_day15["calling_number_normalized"])
lead_ids = set(lead_day15["calling_number_normalized"])

intersection = my_ids & lead_ids
my_only = my_ids - lead_ids
lead_only = lead_ids - my_ids

print(f"My profiles:       {len(my_ids):,}")
print(f"Lead profiles:     {len(lead_ids):,}")
print(f"Intersection:      {len(intersection):,}")
print(f"My only:           {len(my_only):,}")
print(f"Lead only:         {len(lead_only):,}")

print(
    f"\nLead IDs contained in mine: "
    f"{len(intersection) / len(lead_ids) * 100:.2f}%"
)

### Create the three populations

In [ ]:
my_day15["population"] = np.where(
    my_day15["calling_number_normalized"].isin(lead_ids),
    "Lead retained",
    "Lead removed"
)

lead_ids_df = lead_day15[
    lead_day15["calling_number_normalized"].isin(intersection)
].copy()

retained_df = my_day15[
    my_day15["calling_number_normalized"].isin(lead_ids)
].copy()

removed_df = my_day15[
    ~my_day15["calling_number_normalized"].isin(lead_ids)
].copy()

print("Retained:", len(retained_df))
print("Removed:", len(removed_df))

### Verify exact feature alignment

In [ ]:
profile_features = [
    "nb_out_calls",
    "nb_in_calls",
    "nb_called_numbers",
    "avg_out_duration",
    "avg_in_duration",
    "ratio_in_vs_out",
    "ratio_distinct_vs_out",
    "ratio_in_duration_vs_out_duration",
    "dawn_count",
    "day_count",
    "afternoon_count",
    "night_count",
    "ratio_dawn_vs_out",
    "ratio_day_vs_out",
    "ratio_afternoon_vs_out",
    "ratio_night_vs_out",
    "nb_flagged_calls",
    "ratio_flagged_vs_out",
    "nb_distinct_lacs",
    "ratio_distinct_lacs_vs_out",
    "nb_distinct_cell_ids",
    "ratio_distinct_cell_ids_vs_out",
    "avg_delay_per_day",
    "fraud",
]

# Merge
comparison = retained_df[
    ["calling_number_normalized"] + profile_features
].merge(
    lead_day15[
        ["calling_number_normalized"] + profile_features
    ],
    on="calling_number_normalized",
    suffixes=("_mine", "_lead")
)

print("Common profiles:", len(comparison))

# Then
for feature in profile_features:
    mine_col = f"{feature}_mine"
    lead_col = f"{feature}_lead"

    if mine_col not in comparison.columns:
        continue

    equal = (
        comparison[mine_col].fillna(0)
        == comparison[lead_col].fillna(0)
    ).mean()

    print(f"{feature:40s}: {equal * 100:.2f}% identical")

### Feature distribution comparison

In [ ]:
summary_rows = []

for feature in profile_features[:-1]:

    summary_rows.append({
        "feature": feature,

        "my_mean": my_day15[feature].mean(),
        "lead_mean": lead_day15[feature].mean(),
        "removed_mean": removed_df[feature].mean(),

        "my_median": my_day15[feature].median(),
        "lead_median": lead_day15[feature].median(),
        "removed_median": removed_df[feature].median(),

        "my_zero_pct": (
            (my_day15[feature] == 0).mean() * 100
        ),
        "lead_zero_pct": (
            (lead_day15[feature] == 0).mean() * 100
        ),
        "removed_zero_pct": (
            (removed_df[feature] == 0).mean() * 100
        ),
    })

feature_summary = pd.DataFrame(summary_rows)

display(feature_summary)

### Sweetviz — full comparison

In [ ]:
sv_report = sv.compare(
    [retained_df[profile_features], "Lead retained"],
    [removed_df[profile_features], "Lead removed"],
    pairwise_analysis="auto"
)

sv_report.show_html(
    filepath=str(
        DATA_PATH / "diagnostics" / "sweetviz_day15_retained_vs_removed.html"
    ),
    open_browser=True
)

### Zero/non-zero diagnostic

In [ ]:
zero_diagnostic = []

for feature in profile_features[:-1]:

    zero_diagnostic.append({
        "feature": feature,
        "my_zero": (my_day15[feature] == 0).sum(),
        "lead_zero": (lead_day15[feature] == 0).sum(),
        "removed_zero": (removed_df[feature] == 0).sum(),

        "my_zero_pct": (
            (my_day15[feature] == 0).mean() * 100
        ),
        "lead_zero_pct": (
            (lead_day15[feature] == 0).mean() * 100
        ),
        "removed_zero_pct": (
            (removed_df[feature] == 0).mean() * 100
        ),
    })

zero_diagnostic = pd.DataFrame(zero_diagnostic)

display(
    zero_diagnostic.sort_values(
        "removed_zero_pct",
        ascending=False
    )
)

---

### Single-feature threshold search

In [ ]:
numeric_features = [
    f for f in profile_features
    if f != "fraud"
]

def evaluate_filter(df, feature, condition):
    candidate_ids = set(
        df.loc[condition, "calling_number_normalized"]
    )

    overlap = candidate_ids & lead_ids

    precision = (
        len(overlap) / len(candidate_ids)
        if candidate_ids else 0
    )

    recall = (
        len(overlap) / len(lead_ids)
        if lead_ids else 0
    )

    return {
        "feature": feature,
        "candidate_size": len(candidate_ids),
        "overlap": len(overlap),
        "precision": precision,
        "recall": recall,
    }

### Test >= threshold

In [ ]:
threshold_results = []

for feature in numeric_features:

    values = my_day15[feature].dropna()

    thresholds = np.unique(
        np.quantile(
            values,
            np.linspace(0, 1, 21)
        )
    )

    for threshold in thresholds:

        condition = my_day15[feature] >= threshold

        result = evaluate_filter(
            my_day15,
            feature,
            condition
        )

        result["operator"] = ">="
        result["threshold"] = threshold

        threshold_results.append(result)

threshold_results = pd.DataFrame(threshold_results)

display(
    threshold_results
    .sort_values(
        ["precision", "recall"],
        ascending=False
    )
    .head(30)
)

### Test <= threshold

In [ ]:
threshold_results_le = []

for feature in numeric_features:

    values = my_day15[feature].dropna()

    thresholds = np.unique(
        np.quantile(
            values,
            np.linspace(0, 1, 21)
        )
    )

    for threshold in thresholds:

        condition = my_day15[feature] <= threshold

        result = evaluate_filter(
            my_day15,
            feature,
            condition
        )

        result["operator"] = "<="
        result["threshold"] = threshold

        threshold_results_le.append(result)

threshold_results_le = pd.DataFrame(threshold_results_le)

display(
    threshold_results_le
    .sort_values(
        ["precision", "recall"],
        ascending=False
    )
    .head(30)
)

### Test simple non-zero filters

In [ ]:
nonzero_results = []

for feature in numeric_features:

    condition = my_day15[feature] > 0

    result = evaluate_filter(
        my_day15,
        feature,
        condition
    )

    result["operator"] = ">0"
    result["threshold"] = 0

    nonzero_results.append(result)

nonzero_results = pd.DataFrame(nonzero_results)

display(
    nonzero_results.sort_values(
        ["precision", "recall"],
        ascending=False
    )
)

### Rank features by how strongly they separate retained/removed

In [ ]:
separation = []

for feature in numeric_features:

    retained_mean = retained_df[feature].mean()
    removed_mean = removed_df[feature].mean()

    retained_median = retained_df[feature].median()
    removed_median = removed_df[feature].median()

    separation.append({
        "feature": feature,
        "retained_mean": retained_mean,
        "removed_mean": removed_mean,
        "mean_difference": (
            retained_mean - removed_mean
        ),
        "retained_median": retained_median,
        "removed_median": removed_median,
        "median_difference": (
            retained_median - removed_median
        )
    })

separation = pd.DataFrame(separation)

display(
    separation.sort_values(
        "mean_difference",
        key=abs,
        ascending=False
    )
)

### Visualize the strongest candidates

In [ ]:
candidate_features = [
    "nb_out_calls",
    "nb_called_numbers",
    "nb_in_calls",
    "ratio_in_vs_out",
    "ratio_distinct_vs_out",
]

for feature in candidate_features:

    plt.figure(figsize=(8, 5))

    plt.hist(
        retained_df[feature],
        bins=50,
        alpha=0.6,
        label="Lead retained"
    )

    plt.hist(
        removed_df[feature],
        bins=50,
        alpha=0.6,
        label="Lead removed"
    )

    plt.xlabel(feature)
    plt.ylabel("Count")
    plt.title(f"{feature}: retained vs removed")
    plt.legend()
    plt.tight_layout()
    plt.show()

### Check whether X is a random sample

In [ ]:
random_fraction = len(lead_ids) / len(my_ids)

print(
    f"Lead population represents "
    f"{random_fraction * 100:.2f}% of ours."
)

# Generate a random sample of the exact same size
random_sample = my_day15.sample(
    n=len(lead_day15),
    random_state=42
)

random_sample_ids = set(
    random_sample["calling_number_normalized"]
)

# Compare samples
random_comparison = []

for feature in numeric_features:

    random_comparison.append({
        "feature": feature,
        "lead_mean": lead_day15[feature].mean(),
        "random_mean": random_sample[feature].mean(),
        "lead_median": lead_day15[feature].median(),
        "random_median": random_sample[feature].median(),
    })

random_comparison = pd.DataFrame(random_comparison)

display(random_comparison)

### Check ordering-based selection

In [ ]:
my_day15["id_numeric"] = pd.to_numeric(
    my_day15["calling_number_normalized"],
    errors="coerce"
)

lead_day15["id_numeric"] = pd.to_numeric(
    lead_day15["calling_number_normalized"],
    errors="coerce"
)

print("Mine:")
print(my_day15["id_numeric"].describe())

print("\nLead:")
print(lead_day15["id_numeric"].describe())

plt.figure(figsize=(10, 5))

plt.hist(
    my_day15["id_numeric"],
    bins=50,
    alpha=0.5,
    label="Mine"
)

plt.hist(
    lead_day15["id_numeric"],
    bins=50,
    alpha=0.5,
    label="Lead"
)

plt.xlabel("Calling number")
plt.ylabel("Count")
plt.title("Calling-number population")
plt.legend()
plt.tight_layout()
plt.show()

### Check multi-day activity

In [ ]:
daily_profiles = []

for day in range(1, 16):

    path = (
        DATA_PATH
        / "profiles"
        / f"day_{day:02d}.csv"
    )

    df = pd.read_csv(path)

    df["day"] = day

    daily_profiles.append(df)

all_daily_profiles = pd.concat(
    daily_profiles,
    ignore_index=True
)

active_days = (
    all_daily_profiles
    .groupby("calling_number")["day"]
    .nunique()
    .rename("active_days")
)

my_day15 = my_day15.merge(
    active_days,
    on="calling_number",
    how="left"
)

print(
    my_day15.groupby("population")["active_days"]
    .describe()
)

for n in range(1, 16):

    count = (
        my_day15.loc[
            my_day15["active_days"] >= n
        ]
        ["calling_number_normalized"]
        .isin(lead_ids)
        .sum()
    )

    total = (
        my_day15["active_days"] >= n
    ).sum()

    print(
        f"active_days >= {n:2d}: "
        f"{total:6,} profiles | "
        f"{count:6,} lead profiles"
    )

### Check total 15-day call volume

In [ ]:
total_calls = (
    all_daily_profiles
    .groupby("calling_number")["nb_out_calls"]
    .sum()
    .rename("total_out_calls_15d")
)

my_day15 = my_day15.merge(
    total_calls,
    on="calling_number",
    how="left"
)

for threshold in [
    1, 5, 10, 20, 50, 100,
    200, 500, 1000
]:

    candidate = my_day15[
        my_day15["total_out_calls_15d"] >= threshold
    ]

    candidate_ids = set(
        candidate["calling_number_normalized"]
    )

    overlap = candidate_ids & lead_ids

    print(
        f">= {threshold:4} total calls | "
        f"profiles={len(candidate_ids):6,} | "
        f"lead overlap={len(overlap):6,}"
    )

### Check simple combinations

In [ ]:
candidate_conditions = {
    "outgoing_and_incoming": (
        (my_day15["nb_out_calls"] > 0) &
        (my_day15["nb_in_calls"] > 0)
    ),

    "high_outgoing": (
        my_day15["nb_out_calls"] >= 10
    ),

    "high_called_numbers": (
        my_day15["nb_called_numbers"] >= 10
    ),

    "incoming_ratio_low": (
        my_day15["ratio_in_vs_out"] <= 0.5
    ),

    "active_multiple_days": (
        my_day15["active_days"] >= 3
    ),
}

combination_results = []

for name, condition in candidate_conditions.items():

    candidate_ids = set(
        my_day15.loc[
            condition,
            "calling_number_normalized"
        ]
    )

    overlap = candidate_ids & lead_ids

    combination_results.append({
        "experiment": name,
        "candidate_size": len(candidate_ids),
        "lead_size": len(lead_ids),
        "overlap": len(overlap),
        "precision": (
            len(overlap) / len(candidate_ids)
            if candidate_ids else 0
        ),
        "recall": (
            len(overlap) / len(lead_ids)
            if lead_ids else 0
        ),
    })

combination_results = pd.DataFrame(
    combination_results
)

display(
    combination_results.sort_values(
        "recall",
        ascending=False
    )
)

### Final candidate table

In [ ]:
experiments = pd.DataFrame([
    {
        "experiment": "No filter",
        "profiles": len(my_ids),
        "lead_overlap": len(my_ids & lead_ids),
        "precision": len(my_ids & lead_ids) / len(my_ids),
        "recall": len(my_ids & lead_ids) / len(lead_ids),
        "status": "Baseline"
    },

    # Add discovered hypotheses here
])

display(
    experiments.sort_values(
        "recall",
        ascending=False
    )
)